# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlaaSherif-Ibrahim/FLYRANK_AI/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

Working note: this skeleton is filled in for **Lane 2 — Refresh / Content Opportunity Scoring**. Sections are in order; each has a short explanation plus a code cell that backs it with real numbers from the starter dataset.


In [1]:
# Setup — get the repo and data, wherever this kernel starts.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AlaaSherif-Ibrahim/FLYRANK_AI"
REPO_DIR = "FLYRANK_AI"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found - are you at the repo root?"
print("Starter data found. Ready.")


Working dir: C:\Users\alaa\FLYRANK_AI
Starter data found. Ready.


## 1. My lane (or freestyle) and why

**My lane: Lane 2 — Refresh / Content Opportunity Scoring.**

Question this lane answers: *which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?*

Why this one:

- It is the decision this whole track is built around: a content editor has limited review capacity and a backlog of pages with mixed evidence, and needs a ranked, explainable queue to spend that capacity well.
- The starter pipeline (`scripts/01`-`05`) already demonstrates this exact workflow end to end, so I start from a working baseline and a reference implementation instead of a blank page.
- The starter data shows the pool is real and large (cell below) — far more pages than anyone can hand-review.

I may confirm or change this lane until the end of Week 4; this notebook fixes the frame, not the lane.


In [2]:
# The size of the refresh-decision problem on the starter slice.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

declining = int(df["is_declining_label"].sum())
declining_with_demand = int(((df["is_declining_label"] == 1) & (df["impressions_90d"] >= 100)).sum())

print(f"total pages: {len(df):,}")
print(f"pages in the declining bucket: {declining:,}  ({df['is_declining_label'].mean():.1%})")
print(f"declining AND still getting demand (impressions_90d >= 100): {declining_with_demand:,}")


total pages: 30,000
pages in the declining bucket: 16,262  (54.2%)
declining AND still getting demand (impressions_90d >= 100): 13,152


## 2. The question: decision, action, cost of a wrong call

**The decision:** which page does an editor review *first*?

**Who acts on it:** a content editor / reviewer with limited weekly capacity, who chooses what to refresh, expand, protect, prune, or monitor.

**The output:** a ranked review queue — each page scored 0-100, with a suggested action and readable reason codes (e.g. `declining_with_demand`, `stale_visible_page`, `low_ctr_visible_page`).

**What a wrong call costs:**

- **False positive** (a page ranked high that a refresh does not help): editor hours spent on a page that did not need it.
- **False negative** (a declining-with-demand page ranked low): real demand keeps slipping while nobody looks — the quieter, more expensive failure.
- Because capacity is fixed, **precision at the top of the queue** controls both costs: if the top 50 are mostly right, the time is well spent; if they are mostly wrong, it is wasted.

**Why data or ML helps at all:** the pattern is real but too messy to write by hand — many signals (impressions, clicks, CTR, position, age, freshness, engagement) interact and shift over time. The shipped starter run shows the gap: the transparent rule fills about 12 of its top-50 slots correctly; the model fills about 37.


In [3]:
# Why ranking matters: limited reviewer capacity vs the size of the pool.
review_capacity_per_week = 50          # an editor realistically checks ~50 pages/week
weeks = declining_with_demand / review_capacity_per_week
print(f"declining-with-demand pool: {declining_with_demand:,} pages")
print(f"at {review_capacity_per_week} pages/week, one reviewer would need "
      f"{weeks:,.0f} weeks ({weeks/52:.1f} years) to see them all by hand")


declining-with-demand pool: 13,152 pages
at 50 pages/week, one reviewer would need 263 weeks (5.1 years) to see them all by hand


## 3. Quick look at the data (2-3 real numbers)

Three real numbers from the starter dataset make this lane look worth the next 7 weeks:

1. **16,262 of 30,000 pages (54.2%) sit in the declining bucket** — and **13,152 of them still get >= 100 impressions in 90 days**, i.e. they are losing ground but still worth saving. No editor can hand-review that pool.
2. **Median impressions/90d ~ 731 but median clicks/90d = 1** — traffic is extremely heavy-tailed; a ranked queue has to focus on the pages where a change would actually move something.
3. **The shipped starter run already shows the ranking lift:** Precision@50 = 0.240 (hand-rule baseline) -> 0.740 (random forest, client holdout) — about **12 vs 37 of the top-50 review slots correctly filled** (committed `outputs/model_report.md`).

The cell below recomputes 1 and 2 directly from the CSV.


In [4]:
# Three real numbers from the starter dataset that back this lane.
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

n = len(df)
declining = int(df["is_declining_label"].sum())
declining_with_demand = int(((df["is_declining_label"] == 1) & (df["impressions_90d"] >= 100)).sum())
median_impressions = int(df["impressions_90d"].median())
median_clicks = int(df["clicks_90d"].median())

print(f"1) {declining:,}/{n:,} pages ({df['is_declining_label'].mean():.1%}) are in the declining bucket; "
      f"{declining_with_demand:,} of them still have real demand (>=100 impressions / 90d).")
print(f"2) Median impressions/90d = {median_impressions:,}, median clicks/90d = {median_clicks} "
      f"-> heavy-tailed traffic; the queue must focus where change matters.")
print(f"3) Shipped starter run (outputs/model_report.md): Precision@50 = 0.240 (hand-rule baseline) "
      f"-> 0.740 (random forest, client holdout) = about 12 vs 37 of the top-50 review slots correct.")


1) 16,262/30,000 pages (54.2%) are in the declining bucket; 13,152 of them still have real demand (>=100 impressions / 90d).
2) Median impressions/90d = 731, median clicks/90d = 1 -> heavy-tailed traffic; the queue must focus where change matters.
3) Shipped starter run (outputs/model_report.md): Precision@50 = 0.240 (hand-rule baseline) -> 0.740 (random forest, client holdout) = about 12 vs 37 of the top-50 review slots correct.


## 4. Careful words: what I can and can't claim

**What I can claim, with careful words:**

- **Observed / measured** associations on this anonymized starter slice, validated with a client holdout.
- **Decision-support** output: "review these pages first", never "edit this and traffic will recover."
- **Directional** statements about which signals (position, age, freshness, CTR, engagement) track the declining label — after a leakage check.

**What I can NOT claim:**

- That a refresh **caused** a recovery — that needs an experiment or causal design this data cannot provide.
- That these results transfer to the full ~79M-row warehouse without being re-earned there.
- Anything about "Google's algorithm" or about AI rankings / citations.

**Limits I will keep visible:**

- The starter label is a **proxy** (`trend_direction == "down"`, same-window), not a future outcome. A stronger capstone uses prior-window features -> future-window target.
- **Volume floors matter:** the `top_3` position stratum has a median of ~3 impressions/90d — one click swings CTR by tens of points (cell below), so tier comparisons need minimum-volume filters.
- Missingness follows `content_type` (e.g. ~28% of keyword articles lack `word_count`) — a blind `fillna(0)` would leak a category signal; I will add `has_*` flags instead.
- `client_id` / `content_id` are for grouping and client-holdout splits only — never features.


In [5]:
# Why claims need careful wording: even a "great-looking" stratum can be noise.
top3 = df[df["position_tier"] == "top_3"]
print(f"position_tier = top_3: {len(top3):,} rows | median impressions_90d = "
      f"{int(top3['impressions_90d'].median())} | median CTR = {top3['ctr'].median():.3f}%")
print("With ~3 impressions/90d, one click swings CTR by tens of percentage points -> volume floors required.")


position_tier = top_3: 2,321 rows | median impressions_90d = 3 | median CTR = 0.000%
With ~3 impressions/90d, one click swings CTR by tens of percentage points -> volume floors required.


## Self-check

Before submitting, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
